# Lab 0-04 Assignment: Build a Small Agent Specification

Use this notebook after you finish `02_agent_walkthrough.ipynb`.

In this assignment, you will:
- run a plain baseline prompt on the same mini case packet
- edit a small agent specification
- rerun the same model with your bounded agent design
- compare the results

The goal is not to build a perfect agent. The goal is to practice turning a vague model task into a more inspectable workflow.


In [ ]:
import csv
import json
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values
from openai import OpenAI

# Teaching note:
# This setup cell assumes you opened the notebook from this lab folder, then
# loads this lab's .env and the local mini case packet.
LAB_NAME = 'lab0_04_what_is_an_agent'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError(f'Could not find {LAB_NAME}/data')

config = dotenv_values(env_path)
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print("Default model from this lab's .env:", default_model)
print('Ollama endpoint:', ollama_base_url)


## Step 1: Run a Plain Baseline Prompt

Start with a plain review question so you have something to compare against after you design your agent specification.


In [ ]:
baseline_prompt = f"""
Review this mini device-activity case packet and explain what happened and what should be checked next.

Case packet:
{case_packet}
""".strip()

baseline_response = ask_model(default_model, baseline_prompt)

print('Baseline prompt preview:\n')
print(baseline_prompt[:1800])
print('\nBaseline response:')
print('Model:', baseline_response['model'])
print('Time (seconds):', baseline_response['seconds'])
print('-' * 80)
print(baseline_response['raw_text'])


## Step 2: Edit the Agent Specification

Edit the values below before running the next cell.

Keep the task limited. A good Lab 0 agent should have:
- one clear role
- one clear goal
- approved tools
- short memory notes
- a human-review boundary
- a stop condition


### Design Hints

Use these hints only if you get stuck. Keep your agent narrow and cautious.

- `role`: give the agent one limited job, such as reviewing device activity rather than explaining everything in the case
- `goal`: ask for observations, unknowns, and one next review step
- `approved_tools`: keep the tool list limited to the case packet inputs you want the agent to use
- `memory`: include 1-2 short caution notes, not a long paragraph
- `human_review_rule`: tell the agent what should be left to a person
- `stop_condition`: make the agent stop after one structured answer

Weak role example: `analyze everything about this case`

Better role example: `produce a cautious first-pass review of a small device-activity packet`


In [ ]:
student_agent_spec = {
    'agent_name': 'Edit Me: Device Review Agent',
    'role': 'Edit this role so the agent has one limited device-review job.',
    'goal': 'Edit this goal so the agent knows what success looks like.',
    'approved_tools': toolbox,
    'memory': [
        'Edit this memory note.',
        'Edit this memory note.',
    ],
    'required_output_keys': [
        'observations',
        'unknowns',
        'next_step',
        'needs_human_review',
    ],
    'stop_condition': 'Stop after returning the required JSON object.',
    'human_review_rule': 'Edit this rule to say when a human must review the result.',
}

print(json.dumps(student_agent_spec, indent=2))


## Step 3: Run Your Agent Design

The next cell turns your edited agent specification into a prompt, runs the same model again, and checks whether your agent specification looks customized.


In [ ]:
# Teaching note:
# `build_agent_prompt(...)` turns the agent specification fields into one bounded prompt.
# This is where role, memory, rules, and stop condition become agent workflow structure.
def build_agent_prompt(agent: dict, packet: str) -> str:
    tool_lines = '\n'.join(f'- {name}' for name in agent['approved_tools'])
    memory_lines = '\n'.join(f'- {note}' for note in agent['memory'])
    required_keys = ', '.join(agent['required_output_keys'])
    return f"""
You are {agent['agent_name']}.

Role:
{agent['role']}

Goal:
{agent['goal']}

Approved tools:
{tool_lines}

Memory:
{memory_lines}

Rules:
- Use only the information from the provided case packet.
- Do not invent missing events or conclusions.
- Keep observations separate from unknowns.
- Apply this human review rule: {agent['human_review_rule']}
- Return valid JSON only.
- Use exactly these keys: {required_keys}.

Stop condition:
{agent['stop_condition']}

Case packet:
{packet}
""".strip()


remaining_placeholders = []
for key in ['agent_name', 'role', 'goal', 'human_review_rule']:
    if 'Edit this' in student_agent_spec[key] or 'Edit Me' in student_agent_spec[key]:
        remaining_placeholders.append(key)
if any('Edit this' in note for note in student_agent_spec['memory']):
    remaining_placeholders.append('memory')

if remaining_placeholders:
    print('Reminder: customize these fields before trusting the result:', remaining_placeholders)

student_prompt = build_agent_prompt(student_agent_spec, case_packet)
student_response = ask_model(default_model, student_prompt)

print('\nStudent agent prompt preview:\n')
print(student_prompt[:2200])
print('\nStudent agent response:')
print('Model:', student_response['model'])
print('Time (seconds):', student_response['seconds'])
print('-' * 80)
print(student_response['raw_text'])

student_json = None
try:
    student_json = json.loads(clean_json_text(student_response['raw_text']))
except Exception as exc:
    print('\nStudent agent JSON parse error:', exc)

if student_json is not None:
    print('\nParsed student agent JSON:')
    print(json.dumps(student_json, indent=2))

design_check = {
    'role_customized': 'Edit this' not in student_agent_spec['role'],
    'goal_customized': 'Edit this' not in student_agent_spec['goal'],
    'memory_customized': all('Edit this' not in note for note in student_agent_spec['memory']),
    'human_review_rule_customized': 'Edit this' not in student_agent_spec['human_review_rule'],
    'uses_tool_list': bool(student_agent_spec['approved_tools']),
    'has_stop_condition': bool(student_agent_spec['stop_condition']),
}

print('\nAgent-card design check:')
print(json.dumps(design_check, indent=2))


## Reflection Questions

Replace this text with short answers to the questions below.

1. Which part of your agent specification changed the behavior of the model the most?
2. What decision did you leave for human review rather than the agent?
3. Did your agent output feel more inspectable than the baseline response? Why or why not?
4. In one sentence, explain what makes your final design an agent workflow instead of only a chatbot prompt.

## Submission

Save the notebook with:
- the baseline prompt and response
- your edited agent specification
- your student agent response
- the design-check output
- your short reflection answers
